# Connect 4 Live Tournament Helper - Group 3

This notebook is for the live tournament workflow:

- Load your trained Connect 4 model.
- Decide who starts by talking with the opponent.
- On your turn, the notebook recommends a move.
- You tell the opponent your selected column.
- On the opponent's turn, enter the column they tell you.
- The notebook updates the board, validates moves, detects wins/draws, and stores a move log.

Columns are numbered **1 to 7**.


## Part 1 - Helper Code to Set Up Gameplay

In [1]:
import numpy as np
import tensorflow as tf
import pandas as pd
from pathlib import Path
from datetime import datetime

ROWS, COLS = 6, 7
EMPTY, P1, P2 = 0, 1, -1

SEED = 2942
rng = np.random.default_rng(SEED)

print("NumPy:", np.__version__)
print("TensorFlow:", tf.__version__)


c:\Users\Abhiroop Kumar\miniconda3\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


NumPy: 2.1.3
TensorFlow: 2.20.0


In [2]:
# =========================
# Board mechanics
# =========================

def new_board():
    return np.zeros((ROWS, COLS), dtype=np.int8)


def legal_moves(board):
    return [c for c in range(COLS) if board[0, c] == EMPTY]


def is_legal(board, col):
    return isinstance(col, int) and 0 <= col < COLS and board[0, col] == EMPTY


def drop_piece(board, col, player):
    if not is_legal(board, col):
        raise ValueError(f"Illegal move: column {col}. Legal moves are {legal_moves(board)}")

    new = board.copy()
    for r in range(ROWS - 1, -1, -1):
        if new[r, col] == EMPTY:
            new[r, col] = player
            return new

    raise ValueError(f"Column {col} is full.")


def check_win(board, player):
    for r in range(ROWS):
        for c in range(COLS - 3):
            if all(board[r, c + i] == player for i in range(4)):
                return True

    for r in range(ROWS - 3):
        for c in range(COLS):
            if all(board[r + i, c] == player for i in range(4)):
                return True

    for r in range(ROWS - 3):
        for c in range(COLS - 3):
            if all(board[r + i, c + i] == player for i in range(4)):
                return True

    for r in range(3, ROWS):
        for c in range(COLS - 3):
            if all(board[r - i, c + i] == player for i in range(4)):
                return True

    return False


def is_draw(board):
    return len(legal_moves(board)) == 0


def game_status(board):
    if check_win(board, P1):
        return 1
    if check_win(board, P2):
        return -1
    if is_draw(board):
        return 0
    return None


def render(board):
    symbols = {EMPTY: ".", P1: "X", P2: "O"}
    lines = []
    for row in board:
        lines.append(" ".join(symbols[int(v)] for v in row))
    lines.append(" ".join(str(c+1) for c in range(COLS)))
    return "\n".join(lines)


def print_board(board):
    print(render(board))


In [3]:
# Quick sanity check
test_board = new_board()
test_board = drop_piece(test_board, 3, P1)
test_board = drop_piece(test_board, 3, P2)
print_board(test_board)
print("Legal moves:", legal_moves(test_board))


. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . O . . .
. . . X . . .
1 2 3 4 5 6 7
Legal moves: [0, 1, 2, 3, 4, 5, 6]


## Part 2 - Model Loading and Move Recommendation

Supported input shapes:
- CNN: `(None, 6, 7, 2)`
- Transformer/sequence: `(None, 42, 2)`


In [4]:
# =========================
# Load your tournament model
# =========================

MODEL_PATH = "./project3_artifacts/trained_models/pg_improved_riju_m1.keras"  # Change this to your final tournament model path

def load_tournament_model(path):
    try:
        model = tf.keras.models.load_model(path, compile=False, safe_mode=False)
    except TypeError:
        tf.keras.config.enable_unsafe_deserialization()
        model = tf.keras.models.load_model(path, compile=False)

    input_shape = model.input_shape

    if len(input_shape) == 4 and input_shape[1:] == (6, 7, 2):
        model._input_kind = "cnn"
    elif len(input_shape) == 3 and input_shape[1:] == (42, 2):
        model._input_kind = "transformer"
    else:
        raise ValueError(
            f"Unsupported input shape: {input_shape}. "
            "Expected (None, 6, 7, 2) or (None, 42, 2)."
        )

    model._name_tag = Path(path).name
    return model


model = load_tournament_model(MODEL_PATH)
print("Loaded model:", model._name_tag)
print("Input kind:", model._input_kind)
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)


Loaded model: pg_improved_riju_m1.keras
Input kind: cnn
Input shape: (None, 6, 7, 2)
Output shape: (None, 7)


In [ ]:
# =========================
# Model input, probability masking, and recommendation logic
# =========================

def board_to_2ch(board, current_player):
    mine = (board == current_player).astype(np.float32)
    theirs = (board == -current_player).astype(np.float32)
    return np.stack([mine, theirs], axis=-1)


def encode_board(board, current_player, model):
    if model._input_kind == "cnn":
        x = board_to_2ch(board, current_player)
    elif model._input_kind == "transformer":
        x = board_to_2ch(board, current_player).reshape(42, 2)
    else:
        raise ValueError("Unknown model input kind.")

    return x[np.newaxis, ...]


def predict_probs(board, current_player, model):
    x = encode_board(board, current_player, model)
    probs = model(x, training=False).numpy()[0]
    return np.asarray(probs, dtype=np.float32)


def mask_illegal(probs, board):
    legal = legal_moves(board)
    masked = np.zeros(COLS, dtype=np.float32)

    for c in legal:
        masked[c] = probs[c]

    total = masked.sum()
    if total > 0:
        return masked / total

    for c in legal:
        masked[c] = 1.0 / len(legal)

    return masked


def find_winning_move(board, player):
    for col in legal_moves(board):
        after = drop_piece(board, col, player)
        if check_win(after, player):
            return col
    return None


def find_blocking_move(board, player):
    opponent = -player
    for col in legal_moves(board):
        after = drop_piece(board, col, opponent)
        if check_win(after, opponent):
            return col
    return None


def recommend_move(board, current_player, model, force_win=True, block_loss=True):
    if force_win:
        win_col = find_winning_move(board, current_player)
        if win_col is not None:
            probs = np.zeros(COLS, dtype=np.float32)
            probs[win_col] = 1.0
            return win_col, probs, "Immediate winning move"

    if block_loss:
        block_col = find_blocking_move(board, current_player)
        if block_col is not None:
            probs = np.zeros(COLS, dtype=np.float32)
            probs[block_col] = 1.0
            return block_col, probs, "Immediate blocking move"

    raw_probs = predict_probs(board, current_player, model)
    probs = mask_illegal(raw_probs, board)
    col = int(np.argmax(probs))

    return col, probs, "Model argmax recommendation"


def show_recommendation(board, current_player, model):
    col, probs, reason = recommend_move(board, current_player, model)

    prob_table = pd.DataFrame({
        "column": list(range(1, COLS + 1)),
        "legal": [c in legal_moves(board) for c in range(COLS)],
        "probability_after_masking": probs
    })

    print("Recommendation:", col+1)
    print("Reason:", reason)
    display(prob_table)

    return col, probs, reason


## Part 3 - Live Tournament Game

Run the next two cells during the live game.

Helpful inputs:
- Press **Enter** on your turn to accept the model recommendation.
- Type a different legal column to override the model.
- Type `undo` to undo the previous move.
- Type `quit` to stop.


In [ ]:
# =========================
# Live game runner
# =========================

def parse_column_input(text):
    text = str(text).strip().lower()

    if text in {"q", "quit", "exit", "stop"}:
        return "quit"

    if text in {"u", "undo"}:
        return "undo"

    if text == "":
        return ""

    try:
        return int(text) - 1
    except ValueError:
        return None


def winner_name_from_status(status, first_team, second_team):
    if status == 1:
        return first_team
    if status == -1:
        return second_team
    if status == 0:
        return "Draw"
    return None


def live_tournament_game(model):
    our_team = "G3"
    print(f"Your team: {our_team}")
    while True:
        opponent_team = input("Enter opponent team number: ").strip()
        if opponent_team != "":
            break
        print("Opponent team name is required. Please enter a valid value.")

    start_answer = input(f"Does {our_team} start? Enter y/n: ").strip().lower()
    we_start = start_answer in {"y", "yes", "1", "true"}

    if we_start:
        first_team, second_team = our_team, opponent_team
        our_player = P1
    else:
        first_team, second_team = opponent_team, our_team
        our_player = P2

    board = new_board()
    current = P1
    move_log = []
    board_history = [board.copy()]

    print("\nGame setup")
    print("-" * 50)
    print("First player:", first_team, "(X)")
    print("Second player:", second_team, "(O)")
    print(f"{our_team} is:", "X / first player" if our_player == P1 else "O / second player")
    print("-" * 50)
    print_board(board)

    while True:
        status = game_status(board)
        if status is not None:
            winner = winner_name_from_status(status, first_team, second_team)
            print("\nGAME OVER")
            print("-" * 50)
            print("Winner:", winner)
            print("Total moves:", len(move_log))
            print_board(board)

            move_df = pd.DataFrame(move_log)
            if len(move_df) > 0:
                display(move_df)
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                out_path = f"live_game_log_{our_team}_vs_{opponent_team}_{timestamp}.csv"
                move_df.to_csv(out_path, index=False)
                print("Saved move log to:", out_path)

            return {
                "winner": winner,
                "final_board": board,
                "moves": move_df
            }

        current_team = first_team if current == P1 else second_team
        piece = "X" if current == P1 else "O"

        print("\nCurrent board:")
        print_board(board)
        print()
        print(f"Turn {len(move_log) + 1}: {current_team} ({piece})")
        print("Legal moves:", legal_moves(board))

        if current == our_player:
            print("\nYour turn.")
            recommended_col, probs, reason = show_recommendation(board, current, model)
            user_text = input(f"Enter your move column, or press Enter to accept recommendation [{recommended_col + 1}]: ")
            parsed = parse_column_input(user_text)

            if parsed == "quit":
                print("Game stopped by user.")
                break

            if parsed == "undo":
                if len(board_history) > 1:
                    board_history.pop()
                    board = board_history[-1].copy()
                    if move_log:
                        move_log.pop()
                    current = -current
                    print("Undid previous move.")
                    continue
                print("Nothing to undo.")
                continue

            if parsed == "":
                col = recommended_col
            elif isinstance(parsed, int):
                col = parsed
            else:
                print("Invalid input. Please enter a column from 0 to 6.")
                continue

            if not is_legal(board, col):
                print(f"Illegal move: {col}. Legal moves are {legal_moves(board)}")
                continue

            print(f"Tell opponent: {our_team} places in column {col+1}")

            board = drop_piece(board, col, current)
            board_history.append(board.copy())

            move_log.append({
                "move_number": len(move_log) + 1,
                "team": current_team,
                "piece": piece,
                "column": col,
                "source": "model_recommendation" if col == recommended_col else "manual_override",
                "recommendation": recommended_col,
                "recommendation_reason": reason
            })

        else:
            print("\nOpponent's turn.")
            user_text = input(f"Enter the column {current_team} tells you: ")
            parsed = parse_column_input(user_text)

            if parsed == "quit":
                print("Game stopped by user.")
                break

            if parsed == "undo":
                if len(board_history) > 1:
                    board_history.pop()
                    board = board_history[-1].copy()
                    if move_log:
                        move_log.pop()
                    current = -current
                    print("Undid previous move.")
                    continue
                print("Nothing to undo.")
                continue

            if not isinstance(parsed, int):
                print("Invalid input. Please enter a column from 1 to 7.")
                continue

            col = parsed

            if not is_legal(board, col):
                print(f"Illegal move: {col}. Legal moves are {legal_moves(board)}")
                continue

            board = drop_piece(board, col, current)
            board_history.append(board.copy())

            move_log.append({
                "move_number": len(move_log) + 1,
                "team": current_team,
                "piece": piece,
                "column": col,
                "source": "opponent_input",
                "recommendation": None,
                "recommendation_reason": None
            })

        current = -current

    move_df = pd.DataFrame(move_log)
    return {
        "winner": None,
        "final_board": board,
        "moves": move_df
    }


In [7]:
# Run this cell during the live game
live_result = live_tournament_game(model)



Game setup
--------------------------------------------------
First player: G3 (X)
Second player: G11 (O)
G3 is: X / first player
--------------------------------------------------
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
1 2 3 4 5 6 7

Current board:
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
1 2 3 4 5 6 7

Turn 1: G3 (X)
Legal moves: [0, 1, 2, 3, 4, 5, 6]

Your turn.
Recommendation: 4
Reason: Model argmax recommendation


,column,legal,probability_after_masking
0,0,True,3.122687e-12
1,1,True,1.762346e-12
2,2,True,2.958319e-07
3,3,True,9.999996e-01
4,4,True,1.450574e-07
5,5,True,1.048200e-12
6,6,True,3.720921e-13


Tell opponent: G3 places in column 3

Current board:
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . X . . . .
1 2 3 4 5 6 7

Turn 2: G11 (O)
Legal moves: [0, 1, 2, 3, 4, 5, 6]

Opponent's turn.

Current board:
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
O . X . . . .
1 2 3 4 5 6 7

Turn 3: G3 (X)
Legal moves: [0, 1, 2, 3, 4, 5, 6]

Your turn.
Recommendation: 4
Reason: Model argmax recommendation


,column,legal,probability_after_masking
0,0,True,7.010563e-10
1,1,True,3.695954e-11
2,2,True,9.440039e-07
3,3,True,9.999590e-01
4,4,True,4.004445e-05
5,5,True,4.140798e-10
6,6,True,1.875736e-11


Tell opponent: G3 places in column 3

Current board:
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . X . . . .
O . X . . . .
1 2 3 4 5 6 7

Turn 4: G11 (O)
Legal moves: [0, 1, 2, 3, 4, 5, 6]

Opponent's turn.

Current board:
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
O . X . . . .
O . X . . . .
1 2 3 4 5 6 7

Turn 5: G3 (X)
Legal moves: [0, 1, 2, 3, 4, 5, 6]

Your turn.
Recommendation: 3
Reason: Model argmax recommendation


...

Tell opponent: G3 places in column 1

Current board:
. . . . . . .
X . . . . . .
X . O . . . .
O . X . . . .
O X X O . . .
O X X O . . .
1 2 3 4 5 6 7

Turn 14: G11 (O)
Legal moves: [0, 1, 2, 3, 4, 5, 6]

Opponent's turn.

Current board:
. . . . . . .
X . . . . . .
X . O . . . .
O . X O . . .
O X X O . . .
O X X O . . .
1 2 3 4 5 6 7

Turn 15: G3 (X)
Legal moves: [0, 1, 2, 3, 4, 5, 6]

Your turn.
Recommendation: 4
Reason: Immediate blocking move


,column,legal,probability_after_masking
0,0,True,0.0
1,1,True,0.0
2,2,True,0.0
3,3,True,1.0
4,4,True,0.0
5,5,True,0.0
6,6,True,0.0


Tell opponent: G3 places in column 3

Current board:
. . . . . . .
X . X . . . .
X . O . . . .
O . X O . . .
O X X O . . .
O X X O . . .
1 2 3 4 5 6 7

Turn 16: G11 (O)
Legal moves: [0, 1, 2, 3, 4, 5, 6]

Opponent's turn.

GAME OVER
--------------------------------------------------
Winner: G11
Total moves: 16
. . . . . . .
X . X . . . .
X . O O . . .
O . X O . . .
O X X O . . .
O X X O . . .
1 2 3 4 5 6 7


,move_number,team,piece,column,source,recommendation,recommendation_reason
0,1,G3,X,2,manual_override,3.0,Model argmax recommendation
1,2,G11,O,0,opponent_input,NaN,NaN
2,3,G3,X,2,manual_override,3.0,Model argmax recommendation
3,4,G11,O,0,opponent_input,NaN,NaN
4,5,G3,X,1,manual_override,2.0,Model argmax recommendation
5,6,G11,O,0,opponent_input,NaN,NaN
6,7,G3,X,0,model_recommendation,0.0,Immediate blocking move
7,8,G11,O,3,opponent_input,NaN,NaN
8,9,G3,X,1,manual_override,2.0,Model argmax recommendation
9,10,G11,O,3,opponent_input,NaN,NaN


Saved move log to: live_game_log_G3_vs_G11_20260428_153234.csv


## Part 4 - Recovery Tools

In [ ]:
def replay_moves(move_sequence, first_team="Team_First", second_team="Team_Second"):
    """
    Recreate a board from a list of columns.
    Example:
    recovered_board, recovered_log = replay_moves([3, 2, 3, 2], "G3", "G11")
    """
    board = new_board()
    current = P1
    log = []

    for i, col in enumerate(move_sequence, start=1):
        col = int(col) - 1
        if not is_legal(board, col):
            raise ValueError(f"Illegal move at step {i}: column {col}")

        team = first_team if current == P1 else second_team
        board = drop_piece(board, col, current)

        log.append({
            "move_number": i,
            "team": team,
            "piece": "X" if current == P1 else "O",
            "column": col
        })

        if game_status(board) is not None:
            break

        current = -current

    print_board(board)
    return board, pd.DataFrame(log)


# Example:
# recovered_board, recovered_log = replay_moves([3, 2, 3, 2], "G3", "G11")
# display(recovered_log)


In [ ]:
# Recommend from a recovered board
# current_player = P1  # or P2
# show_recommendation(recovered_board, current_player, model)


## Part 5 - Result Summary

In [ ]:
try:
    print("Winner:", live_result["winner"])
    print("\nFinal board:")
    print_board(live_result["final_board"])
    print("\nMove log:")
    display(live_result["moves"])
except NameError:
    print("Run the live game cell first.")
